# Cygnus tiered lead pursuit on a Google Colab **CPU** runtime

**What this is.** The repository's official notebook for taking targets through the lead-proving tiers
(`src/cygnus/campaign/tiers.py`, `docs/CAMPAIGNS.md` *Tier gating*). Each target earns tools from its own
sky record: **T0** screen, **T1** triage, **T2** vet, **T3** discriminate (heavy, budgeted), **T4** confirm
(needs the user's go-ahead). A target never runs a tier it has not earned, and `MAX_TIER` caps the whole run.
Most targets stop at T0, so a big list stays cheap.

**What it is not.** It submits nothing to any catalogue or observer, publishes nothing, deploys nothing,
changes no science threshold, and never selects a GPU/TPU. It never overwrites a committed record: targets
that already have one are only tiered and vetted inside this VM's clone, and the outputs ship to Drive.

**Drive.** Uses `drive.mount()` (Option B, `docs/STORAGE.md`). Everything is written under
`Cygnus/colab_runs/<RUN_ID>/` only; nothing outside `Cygnus/` is read or written. Colab-written folders are
visible on the workstation through Google Drive for desktop (checked 2026-09-30); the `drive.file` rclone token still cannot see them, so verify with
`python -m cygnus.storage check colab_runs/<RUN_ID>`.

**Credentials.** None entered, printed or stored. Clear outputs before committing this notebook.

**Resumable.** Re-running from the top fetches the pin, restores the ledger and progress from Drive, and skips
finished targets. Sessions end on idle or lifetime limits; a checkpoint ships after every target.

In [ ]:
# Step 0. Configuration. Edit only this cell.
RUN_ID = 'SET_RUN_ID'                 # e.g. leads-2026-10-01 (a folder name under Cygnus/colab_runs/)
TARGETS = []                          # campaign ids, e.g. ['toi-224-01', 'toi-2666-01', 'toi-3500-02']
MAX_TIER = 'T2'                       # cap for this run: T0..T2 run here; T3 needs BUDGET_MB; T4 is refused here
BUDGET_MB = None                      # T3 only: total download budget in MB you approve for this run
FULL_TESTS = False                    # True runs the whole offline suite (~10 min); False runs the fast gate
PINNED_COMMIT = 'SET_PINNED_COMMIT'   # full 40-hex SHA that contains the tier + native-tool modules (pushed!)
import re
if RUN_ID.startswith('SET_') or not re.fullmatch(r'[A-Za-z0-9._-]+', RUN_ID):
    raise ValueError('Set RUN_ID to a plain folder name')
if not TARGETS:
    raise ValueError('Set TARGETS to a list of campaign ids')
if MAX_TIER not in ('T0', 'T1', 'T2', 'T3'):
    raise ValueError('MAX_TIER is T0..T3; T4 is never run from this notebook (needs the user go-ahead)')
if MAX_TIER == 'T3' and not BUDGET_MB:
    raise ValueError('T3 downloads spectra/cutouts: state BUDGET_MB before enabling it')
if not re.fullmatch(r'[0-9a-f]{40}', PINNED_COMMIT):
    raise ValueError('PINNED_COMMIT must be a full 40-hex SHA of a pushed commit')

In [ ]:
# Step 1. CPU-only check and the Drive mount.
import os, shutil, subprocess
from pathlib import Path
acc = shutil.which('nvidia-smi')
if (acc and subprocess.run([acc], capture_output=True).returncode == 0) or os.environ.get('COLAB_TPU_ADDR'):
    raise RuntimeError('GPU/TPU runtime detected: switch to CPU. Nothing here needs an accelerator.')
from google.colab import drive
drive.mount('/content/drive')
CYGNUS_DRIVE_ROOT = Path('/content/drive/MyDrive/Cygnus')
if not CYGNUS_DRIVE_ROOT.is_dir():
    raise FileNotFoundError(f'{CYGNUS_DRIVE_ROOT} not found: the Cygnus/ folder must exist on this Drive (nothing else is touched)')
print('Cygnus root:', CYGNUS_DRIVE_ROOT, '| CPUs:', os.cpu_count())

In [ ]:
# Step 2. Pinned checkout, install, gate. Nothing runs before the tests pass.
import subprocess, sys
REPO_URL = 'https://github.com/mapsugui/ad_astra'          # public: no credential needed
REPO_DIR = Path('/content/ad_astra')
if not (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', 'clone', '--quiet', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--quiet', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--quiet', '--detach', PINNED_COMMIT], check=True)
head = subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
if head != PINNED_COMMIT:
    raise RuntimeError(f'checkout {head} != pin {PINNED_COMMIT}: refusing to run unrecorded code')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', f'{REPO_DIR}[test,mast,campaign]'], check=True)
if str(REPO_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_DIR / 'src'))        # kernel started before the .pth existed
gate = ['-q'] if FULL_TESTS else ['-q', 'tests/test_native_lead_tools.py', 'tests/test_skyrecord.py', 'tests/test_campaign.py']
if subprocess.run([sys.executable, '-m', 'pytest', *gate], cwd=str(REPO_DIR)).returncode != 0:
    raise RuntimeError('tests failed on this runtime: fix the environment or re-pin before running any science')
print('gate passed:', 'full offline suite' if FULL_TESTS else 'fast gate', '| commit', head)

In [ ]:
# Step 3. VM scratch (never the mount), and resume state from Drive.
import json
from cygnus.colab import sha256_file
SCRATCH = Path('/content/scratch'); SCRATCH.mkdir(parents=True, exist_ok=True)
RUN_DIR = CYGNUS_DRIVE_ROOT / 'colab_runs' / RUN_ID
LEDGER = REPO_DIR / 'state' / 'ledger.sqlite'; LEDGER.parent.mkdir(parents=True, exist_ok=True)
ENV = {**os.environ, 'CYGNUS_SCRATCH': str(SCRATCH), 'CYGNUS_LEDGER': str(LEDGER), 'CYGNUS_RATE_S': '0.5'}
if (RUN_DIR / 'ledger.sqlite').is_file():
    for suf in ('', '-wal', '-shm'):
        src = RUN_DIR / f'ledger.sqlite{suf}'
        if src.is_file():
            shutil.copyfile(src, Path(str(LEDGER) + suf))
    print('restored companion ledger from', RUN_DIR)
DONE_FILE = RUN_DIR / 'done.json'
DONE = json.loads(DONE_FILE.read_text()) if DONE_FILE.is_file() else {}
print('already finished in a previous session:', sorted(DONE) or 'none')

In [ ]:
# Step 4. Tier board: what each target has earned, from its committed sky record. Read-only.
import json, subprocess
from cygnus.campaign.tiers import decide, TIERS


def committed_record(cid):
    r = subprocess.run(['git', '-C', str(REPO_DIR), 'show', f'{PINNED_COMMIT}:campaigns/{cid}/sky_record.json'],
                       capture_output=True, text=True)
    return json.loads(r.stdout) if r.returncode == 0 else None


BOARD = {}
print(f"{'target':<16}{'earned':<8}{'run at':<8}{'cost':<8}blockers")
for cid in TARGETS:
    existing = committed_record(cid)
    rec = existing or {'status': 'draft', 'outcome': 'not_run', 'checks': []}
    rej = subprocess.run(['git', '-C', str(REPO_DIR), 'cat-file', '-e', f'{PINNED_COMMIT}:campaigns/{cid}/REJECTION.md']).returncode == 0
    d = decide(rec, rejection_note=rej)
    cap = TIERS.index(MAX_TIER)
    run_at = d.tier if d.tier == 'closed' else TIERS[min(TIERS.index(d.tier), cap)]
    BOARD[cid] = {'earned': d.tier, 'run_at': run_at, 'record_exists': existing is not None,
                  'status': rec.get('status'), 'decision': d.as_dict()}
    print(f"{cid:<16}{d.tier:<8}{run_at:<8}{d.cost:<8}{'; '.join(d.blockers) or '-'}")
closed = [c for c, b in BOARD.items() if b['earned'] == 'closed']
if closed:
    print('closed (rejection note only, no analysis):', closed)

In [ ]:
# Step 5. Run what each target has earned, capped by MAX_TIER. One target at a time, checkpoint after each.
#   T0/T1: `campaign run` -- ONLY for a target with no committed non-draft record (never rewrite a committed one).
#   T2   : `campaign vet` (TPF difference image, neighbours) in this clone; outputs ship to Drive, not into the repo.
#   T3   : planned tools are printed, not run: they are library functions without campaign steps yet
#          (docs/NATIVE_SUPPORT_GAPS_2026-09-30.md). Run them deliberately in the next cell if approved.
import subprocess, sys, sqlite3


def checkpoint(cid):
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    with sqlite3.connect(LEDGER) as s, sqlite3.connect(RUN_DIR / 'ledger.sqlite') as t:
        s.backup(t)
    out = REPO_DIR / 'campaigns' / cid
    if out.is_dir():
        shutil.copytree(out, RUN_DIR / 'campaigns' / cid, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns('normalized_series.csv', '__pycache__'))
    DONE[cid] = BOARD[cid]
    DONE_FILE.write_text(json.dumps(DONE, indent=1, sort_keys=True) + '\n')


def sh(args, timeout=3600):
    p = subprocess.run([sys.executable, '-m', *args], cwd=str(REPO_DIR), env=ENV, capture_output=True,
                       text=True, timeout=timeout)
    print(p.stdout[-1500:], p.stderr[-800:] if p.returncode else '')
    return p.returncode


PLAN = {}
for cid, b in BOARD.items():
    if cid in DONE:
        print(cid, 'already done'); continue
    tier = b['run_at']
    if tier == 'closed':
        continue
    spec = f'campaigns/{cid}.yaml'
    if not (REPO_DIR / spec).is_file():
        print(cid, 'no spec in the pinned commit: create it locally (`campaign new`) and commit before running')
        continue
    rc = 0
    if b['record_exists'] and b['status'] not in (None, 'draft'):
        print(cid, f'{tier}: committed record exists -> screen NOT re-run (frozen); vetting only if T2+')
    else:
        rc = sh(['cygnus.campaign', 'run', spec, '--ledger', str(LEDGER)])
    if rc == 0 and tier in ('T2', 'T3'):
        rc = sh(['cygnus.campaign', 'vet', spec, '--ledger', str(LEDGER)])
    if tier == 'T3':
        PLAN[cid] = b['decision']['allowed'][-5:]
        print(cid, 'T3 planned tools (not run):', PLAN[cid])
    print(cid, 'exit', rc)
    if rc == 0:
        checkpoint(cid)

In [ ]:
# Step 6. T3 discriminate -- only for targets that earned T3 AND MAX_TIER == 'T3' AND BUDGET_MB set.
# Deliberately manual: fetch with cygnus.multi.archives (eso names=[...], koa), track bytes against BUDGET_MB,
# record every attempt with cygnus.ingest.access.ProductAccess (401/timeout are limits, not absence), and gate
# any derived RV behind cygnus.analysis.spectra.precision_gate / require_gate. Template:
#
#   from cygnus.multi.archives import base
#   eso = base.get('eso'); refs = eso.discover(target, names=['<exact name>'])
#
# T4 (spectral extraction, calibrations, follow-up proposal) is not run from Colab without the user's go-ahead.
t3 = [c for c, b in BOARD.items() if b['run_at'] == 'T3']
print('T3-eligible under this cap:' if MAX_TIER == 'T3' else 'T3 disabled (MAX_TIER < T3):', t3)

In [ ]:
# Step 7. Archive health from this egress IP, then ship: manifest, read-back verification, LOCATION.json.
import datetime
from cygnus.colab import build_manifest, write_manifest, verify_manifest
h = subprocess.run([sys.executable, '-m', 'cygnus.multi', 'archives', '--check', 'all'], cwd=str(REPO_DIR), env=ENV,
                   capture_output=True, text=True, timeout=3600)
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'archives_check.txt').write_text(f'exit {h.returncode}\n{h.stdout}\n{h.stderr}\n', encoding='utf-8')
(RUN_DIR / 'tier_board.json').write_text(json.dumps({
    'run': RUN_ID, 'commit': PINNED_COMMIT, 'max_tier': MAX_TIER, 'budget_mb': BUDGET_MB, 'board': BOARD,
    'planned_t3': PLAN, 'utc': datetime.datetime.now(datetime.timezone.utc).isoformat()},
    indent=1, sort_keys=True) + '\n', encoding='utf-8')
shipped = ['ledger.sqlite', 'archives_check.txt', 'tier_board.json', 'done.json'] + \
          [f'campaigns/{c}' for c in DONE if (RUN_DIR / 'campaigns' / c).is_dir()]
manifest = build_manifest(RUN_DIR, [p for p in shipped if (RUN_DIR / p).exists()])
MANIFEST = write_manifest(RUN_DIR / 'MANIFEST.sha256', manifest,
                          comment=f'{RUN_ID}; commit {PINNED_COMMIT}; {len(manifest)} files')
problems = verify_manifest(MANIFEST, RUN_DIR, ignore_extra=['_probe/*'])
print('verified from the mount:', len(manifest), 'files' if not problems else problems)
from cygnus.storage import Route, folder_id, make_entry, upsert
R = Route('path', str(CYGNUS_DRIVE_ROOT), 'Colab mount')
LOC = make_entry(f'colab_runs/{RUN_ID}', kind='colab_run', writer=R.label,
                 producer='notebooks/cygnus_lead_colab.ipynb', commit=PINNED_COMMIT,
                 manifest_sha256=sha256_file(MANIFEST), files=len(manifest),
                 drive_folder_id=folder_id(R, f'colab_runs/{RUN_ID}'),
                 notes=f'tiered lead pursuit, cap {MAX_TIER}; companion ledger, never merged')
LOC['visible_via'] = {R.label: {'state': 'visible' if not problems else 'not_visible', 'utc': LOC['created_utc'],
                                'note': f'{len(manifest)} files read back from the mount'}}
(RUN_DIR / 'LOCATION.json').write_text(json.dumps(LOC, indent=1, sort_keys=True) + '\n', encoding='utf-8')
upsert(LOC)
print('Open:', LOC['links']['folder'] or LOC['links']['search'])
print(json.dumps(LOC, sort_keys=True, ensure_ascii=False))

## Next: on the workstation

1. `python -m cygnus.storage add <path to LOCATION.json>` (the workstation sees the folder through Google Drive for
   desktop), then `python -m cygnus.storage check colab_runs/<RUN_ID>`.
2. Verify `MANIFEST.sha256` (`cygnus.colab.verify_manifest`); an empty problem list is the only pass.
3. Read `tier_board.json` and the shipped vetting outputs; **do not copy `campaigns/<id>/` over committed records** (their
   ledger `run_id`s live in the local ledger; the Colab ledger is a named companion, never merged).
4. T3 also needs `budget` and `discriminating_question` in the campaign spec; a run below the earned tier can only be forced with a
   recorded `tier_override` (`{tier, reason, approved_by, date}`), never silently.
5. Update the sky record / report through the normal review (`docs/AGENT_RUNBOOK.md`), then
   `python -m cygnus.campaign reconcile <id> --note ...` if a lead changed state.